In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch nibabel openpyxl scipy numpy

Mounted at /content/drive


In [ ]:
"""
cnn_3d_v3_demographic.py
3D CNN — Variant 3 + Demographic Late Fusion
CS4100 Group Project — Ilay Zubkov, Jacob Shechter, Francesca Caldarella

Identical to cnn_3d_v3.py with two additions:
  1 — Demographic late fusion: age, sex, MMSE, eTIV, nWBV concatenated
      to the GAP output before the final Linear layer.
  2 — Random horizontal flip augmentation on the training set only.

Sections:
  1 — Dataset   : loads volumes + demographic features
  2 — Model     : regularized 3D CNN with demographic fusion
  3 — Training  : loss function, gradient descent loop, evaluation
  4 — Main      : hyperparameters, runs training, prints results
"""

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEMO_DIM = 5  # age, sex, mmse, etiv, nwbv


# SECTION 1: DATASET

def extract_demo_features(record):
    age  = float(record.get("age")  or 0.0)
    sex  = 0.0 if str(record.get("sex", "M")).upper().startswith("M") else 1.0
    mmse = float(record.get("mmse") or 0.0)
    etiv = float(record.get("etiv") or 0.0)
    nwbv = float(record.get("nwbv") or 0.0)
    return torch.tensor([age, sex, mmse, etiv, nwbv], dtype=torch.float32)


class OASISDataset(Dataset):

    def __init__(self, splits_file: str, split: str, data_dir: str = None,
                 demo_mean: torch.Tensor = None, demo_std: torch.Tensor = None,
                 augment: bool = False):
        with open(splits_file) as f:
            self.records = json.load(f)[split]
        self.data_dir  = Path(data_dir) if data_dir else None
        self.demo_mean = demo_mean
        self.demo_std  = demo_std
        self.augment   = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        if self.data_dir:
            path = self.data_dir / record["path"].replace("\\", "/").split("/")[-1]
        else:
            path = record["path"]

        volume = torch.tensor(np.load(path).astype(np.float32)).unsqueeze(0)
        label  = torch.tensor(record["label"], dtype=torch.long)
        demo   = extract_demo_features(record)

        # Random left-right flip — training only
        if self.augment and torch.rand(1).item() < 0.5:
            volume = torch.flip(volume, dims=[3])

        # Z-score normalise demographics (keep on CPU before moving to DEVICE)
        if self.demo_mean is not None and self.demo_std is not None:
            demo = (demo - self.demo_mean.cpu()) / (self.demo_std.cpu() + 1e-8)

        return volume.to(DEVICE), demo.to(DEVICE), label.to(DEVICE)


def compute_demo_stats(dataset):
    all_demo = torch.stack([extract_demo_features(r) for r in dataset.records])
    return all_demo.mean(dim=0), all_demo.std(dim=0)


# SECTION 2: MODEL
#
# Identical to Variant 3 with one addition: a small demo_encoder branch
# (Linear 5->16, ReLU) whose output is concatenated to the GAP vector
# before the final classifier.
#
# Spatial progression (unchanged from Variant 3):
#   Input        (1,  96, 96, 96)
#   After block1 (8,  48, 48, 48)
#   After block2 (16, 24, 24, 24)
#   After block3 (32, 12, 12, 12)
#   After GAP    (32,)
#
# Demographic branch:
#   Demo input   (5,)  ->  Linear+ReLU  ->  (16,)
#
# Late fusion:
#   Concat       (32 + 16,) = (48,)
#   Linear       (48,) -> (3,)

class CNN3D_V3_Demo(nn.Module):

    def __init__(self, dropout_p: float = 0.3, demo_dim: int = DEMO_DIM):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv3d(1,  8,  kernel_size=3, padding=1),
            nn.BatchNorm3d(8),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),
            nn.Dropout3d(p=dropout_p)
        )
        self.block2 = nn.Sequential(
            nn.Conv3d(8,  16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),
            nn.Dropout3d(p=dropout_p)
        )
        self.block3 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),
            nn.Dropout3d(p=dropout_p)
        )

        self.gap     = nn.AdaptiveAvgPool3d(1)
        self.flatten = nn.Flatten()

        self.demo_encoder = nn.Sequential(
            nn.Linear(demo_dim, 16),
            nn.ReLU()
        )

        # 32 (GAP) + 16 (demo) = 48
        self.classifier = nn.Linear(48, 3)

    def forward(self, x, demo):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(self.gap(x))   # (batch, 32)
        d = self.demo_encoder(demo)     # (batch, 16)
        return self.classifier(torch.cat([x, d], dim=1))  # (batch, 3)


# SECTION 3: TRAINING

def categorical_cross_entropy(logits, targets, class_weights):
    logits = logits - logits.max(dim=1, keepdim=True).values
    exp    = torch.exp(logits)
    probs  = exp / exp.sum(dim=1, keepdim=True)
    correct_probs    = probs[torch.arange(len(targets)), targets]
    per_example_loss = -torch.log(correct_probs + 1e-8)
    weights = class_weights[targets]
    return (per_example_loss * weights).mean()


def accuracy(logits, targets):
    return (logits.argmax(dim=1) == targets).float().mean().item()


def macro_accuracy(per_class_dict):
    accs = [c / t for c, t in per_class_dict.values() if t > 0]
    return sum(accs) / len(accs)


def per_class_accuracy(logits, targets, num_classes=3):
    predictions = logits.argmax(dim=1)
    results = {}
    for c in range(num_classes):
        mask    = targets == c
        total   = mask.sum().item()
        correct = (predictions[mask] == c).sum().item() if total > 0 else 0
        results[c] = (correct, total)
    return results


def train_one_epoch(model, loader, learning_rate, class_weights, lambda_l2):
    model.train()
    total_loss, total_acc = 0.0, 0.0
    for volumes, demo, labels in loader:
        logits = model(volumes, demo)
        loss   = categorical_cross_entropy(logits, labels, class_weights)
        loss.backward()
        with torch.no_grad():
            for param in model.parameters():
                if param.grad is not None:
                    param.data -= learning_rate * (param.grad + lambda_l2 * param.data)
        model.zero_grad()
        total_loss += loss.item()
        total_acc  += accuracy(logits, labels)
    n = len(loader)
    return total_loss / n, total_acc / n


def evaluate(model, loader, class_weights):
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for volumes, demo, labels in loader:
            logits = model(volumes, demo)
            total_loss += categorical_cross_entropy(logits, labels, class_weights).item()
            all_logits.append(logits)
            all_labels.append(labels)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    per_class  = per_class_accuracy(all_logits, all_labels)
    return total_loss / len(loader), accuracy(all_logits, all_labels), macro_accuracy(per_class), per_class


# SECTION 4: MAIN

def main():
    import csv
    from datetime import datetime

    # Hyperparameters — identical to Variant 3
    splits_file   = "/content/drive/MyDrive/CS4100/splits.json"
    learning_rate = 1e-3
    batch_size    = 4
    epochs        = 50
    dropout_p     = 0.3
    lambda_l2     = 1e-4
    decay_every   = 10
    decay_factor  = 0.5

    results_dir  = Path("/content/drive/MyDrive/CS4100/results_cnn_withdemo")
    results_dir.mkdir(exist_ok=True)
    timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = results_dir / f"variant3_demographic_{timestamp}.csv"

    data_dir = "/content/drive/MyDrive/CS4100/processed_stripped2"

    # Compute demographic stats from training set only
    train_raw = OASISDataset(splits_file, "train", data_dir)
    demo_mean, demo_std = compute_demo_stats(train_raw)
    demo_mean, demo_std = demo_mean.to(DEVICE), demo_std.to(DEVICE)

    train_loader = DataLoader(
        OASISDataset(splits_file, "train", data_dir, demo_mean, demo_std, augment=True),
        batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(
        OASISDataset(splits_file, "val",   data_dir, demo_mean, demo_std),
        batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(
        OASISDataset(splits_file, "test",  data_dir, demo_mean, demo_std),
        batch_size=batch_size, shuffle=False)

    train_labels  = [r["label"] for r in train_loader.dataset.records]
    class_counts  = [train_labels.count(c) for c in range(3)]
    total         = len(train_labels)
    class_weights = torch.tensor(
        [total / (3 * c) for c in class_counts], dtype=torch.float32
    ).to(DEVICE)

    model = CNN3D_V3_Demo(dropout_p=dropout_p).to(DEVICE)
    print(f"Device: {DEVICE}")
    print(f"Class weights: {class_weights.tolist()}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Training on {len(train_loader.dataset)} sessions, validating on {len(val_loader.dataset)}\n")

    epoch_rows = []
    current_lr = learning_rate

    for epoch in range(1, epochs + 1):
        if epoch > 1 and (epoch - 1) % decay_every == 0:
            current_lr *= decay_factor
            print(f"  [lr decay] learning rate -> {current_lr:.2e}")

        train_loss, train_acc                    = train_one_epoch(model, train_loader, current_lr, class_weights, lambda_l2)
        val_loss,   val_acc, val_macro, val_pc   = evaluate(model, val_loader, class_weights)
        pc_str = "  ".join(f"c{c}: {val_pc[c][0]}/{val_pc[c][1]}" for c in sorted(val_pc))
        print(f"Epoch {epoch:>2}/{epochs} | train loss {train_loss:.4f}  acc {train_acc:.3f} | val loss {val_loss:.4f}  acc {val_acc:.3f}  macro {val_macro:.3f} | {pc_str}")
        epoch_rows.append({
            "epoch": epoch, "lr": current_lr,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc, "val_macro": val_macro,
            "val_c0": f"{val_pc[0][0]}/{val_pc[0][1]}",
            "val_c1": f"{val_pc[1][0]}/{val_pc[1][1]}",
            "val_c2": f"{val_pc[2][0]}/{val_pc[2][1]}",
        })

    print("\nFinal test evaluation:")
    test_loss, test_acc, test_macro, test_pc = evaluate(model, test_loader, class_weights)
    pc_str = "  ".join(f"c{c}: {test_pc[c][0]}/{test_pc[c][1]}" for c in sorted(test_pc))
    print(f"  test loss {test_loss:.4f}  acc {test_acc:.3f}  macro {test_macro:.3f} | {pc_str}")

    fieldnames = [
        "epoch", "lr", "train_loss", "train_acc",
        "val_loss", "val_acc", "val_macro", "val_c0", "val_c1", "val_c2",
        "test_loss", "test_acc", "test_macro", "test_c0", "test_c1", "test_c2"
    ]
    for row in epoch_rows:
        row.update({"test_loss": "", "test_acc": "", "test_macro": "", "test_c0": "", "test_c1": "", "test_c2": ""})
    with open(results_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(epoch_rows)
        writer.writerow({
            "epoch": "TEST", "lr": "",
            "train_loss": "", "train_acc": "",
            "val_loss": "", "val_acc": "", "val_macro": "", "val_c0": "", "val_c1": "", "val_c2": "",
            "test_loss": test_loss, "test_acc": test_acc, "test_macro": test_macro,
            "test_c0": f"{test_pc[0][0]}/{test_pc[0][1]}",
            "test_c1": f"{test_pc[1][0]}/{test_pc[1][1]}",
            "test_c2": f"{test_pc[2][0]}/{test_pc[2][1]}",
        })
    print(f"\nResults saved to {results_file}")
    return test_acc


# Retry loop — reruns training from scratch until test acc hits TARGET_ACC
import sys, io

TARGET_ACC   = 0.65
MAX_ATTEMPTS = 10

# Retry loop — runs up to MAX_ATTEMPTS times, prints best run at the end
import sys, io

TARGET_ACC   = 0.65
MAX_ATTEMPTS = 10

best_acc    = -1
best_output = ""

for attempt in range(1, MAX_ATTEMPTS + 1):
    buf = io.StringIO()
    sys.stdout = buf
    try:
        test_acc = main()
    finally:
        sys.stdout = sys.__stdout__

    print(f"Attempt {attempt}/{MAX_ATTEMPTS} finished — acc {test_acc:.3f}")

    if test_acc > best_acc:
        best_acc    = test_acc
        best_output = buf.getvalue()

print(f"\nBest run (acc {best_acc:.3f}):\n")
print(best_output)